# Convergência Estatística Aplicada à Precificação de Opções via Modelo BinomialEste projeto usa a **precificação de uma opção de compra europeia** (via modelo binomial e simulação de Monte Carlo) como um laboratório prático para demonstrar empiricamente três teoremas fundamentais da probabilidade:- **Lei Fraca dos Grandes Números (LFGN)** — convergência em probabilidade- **Lei Forte dos Grandes Números (LFGN forte)** — convergência quase certa- **Teorema Central do Limite (TLC)** — incluindo o Teorema de De Moivre–LaplaceOs parâmetros do modelo são estimados a partir de **dados reais de mercado**: preço e volatilidade da ação PETR4 (via Yahoo Finance) e taxa livre de risco Selic (via API do Banco Central).**Roteiro:**| Módulo | Conteúdo ||---|---|| 0 | Coleta de dados, parâmetros e funções base || 1 | LFGN — convergência em probabilidade || 2 | LFGN forte — convergência quase certa || 3 | TLC — De Moivre–Laplace e intervalos de confiança || 4 | Convergência do Binomial para Black–Scholes |

## Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, norm
import yfinance as yf
import requests as rt
import math as M

# estilo padrão dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')

## Módulo 0 — Coleta de dados, parâmetros e funções baseAntes de simular qualquer coisa, precisamos dos seis parâmetros do modelo binomial. Três vêm de dados reais de mercado ($S_0$, $\sigma$, $r$) e três são escolhas de cenário ($K$, $T$, $n_{steps}$).

### Download do histórico de preços da PETR4 (2 anos)

In [ ]:
dados = yf.download("PETR4.SA", period="2y")
dados[['Close']].tail()

### Volatilidade anualizada ($\sigma$)A volatilidade histórica é o desvio padrão dos log-retornos diários, anualizado por $\sqrt{252}$ (dias úteis no ano).

In [ ]:
close = dados['Close'].squeeze()
razao = close / close.shift(1)          # razão entre fechamento de hoje e o do dia anterior
log_retorno = np.log(razao).dropna()    # log-retorno diário (descarta o 1º dia, sem anterior)

dp = log_retorno.std()                  # desvio padrão amostral (diário)
dp_anual = dp * np.sqrt(252)            # volatilidade anualizada

print(f"Volatilidade anualizada (sigma): {dp_anual:.4f}")

### Taxa livre de risco ($r$) — Selic via API do Banco CentralUsamos a série 11 (Selic) do sistema SGS do Banco Central, pegando apenas o valor mais recente.

In [ ]:
codigo = 11
n = 1
url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados/ultimos/{n}?formato=json'
resposta = rt.get(url)
selic = resposta.json()
r = float(selic[0]['valor'])            # taxa livre de risco (anual)

print(f"Taxa Selic (r): {r:.4f}  ({selic[0]['data']})")

### Parâmetros de cenário- $S_0$: preço atual da ação (último fechamento)- $K$: preço de exercício (strike), escolhido igual a $S_0$ arredondado — opção *at the money*- $T$: tempo até o vencimento, em anos (0,25 = 3 meses)- $n_{steps}$: número de passos da árvore binomial

In [ ]:
s0 = close.iat[-1]        # preço atual da ação
strike = round(s0)        # preço de exercício (arredondado)
T = 0.25                  # tempo de vencimento (em anos)
n_steps = 200             # número de passos da árvore binomial

print(f"S0 = {s0:.2f} | K = {strike} | T = {T} | n_steps = {n_steps}")

### Parâmetros derivados: $u$, $d$, $p$A cada passo de tamanho $\Delta t = T/n_{steps}$, o preço sobe por um fator $u$ ou desce por $d$, com probabilidade neutra ao risco $p$:$$u = e^{\sigma\sqrt{\Delta t}}, \qquad d = \frac{1}{u}, \qquad p = \frac{e^{r\Delta t} - d}{u - d}$$

In [ ]:
def outputs(vol):
    deltaT = T / n_steps
    u = M.exp(vol * M.sqrt(deltaT))
    d = 1 / u
    p = (M.exp(r * deltaT) - d) / (u - d)
    return u, d, p

u, d, p = outputs(dp_anual)
print(f"u = {u:.6f} | d = {d:.6f} | p = {p:.6f}")

### Função de simulação de uma trajetóriaComo o preço final depende apenas de **quantas vezes** o preço subiu (e não da ordem), sorteamos o número de subidas $k$ diretamente de uma distribuição Binomial, em vez de simular passo a passo.$$S_n = S_0 \cdot u^k \cdot d^{\,n_{steps}-k}, \qquad \text{payoff} = \max(S_n - K,\ 0)$$

In [ ]:
def simulacao(s, P, sub, des, si, K_strike):
    k = np.random.binomial(s, P)                 # número de subidas (sorteado)
    Sn = si * (sub ** k) * (des ** (s - k))      # preço final da ação
    payoff = max(Sn - K_strike, 0)               # payoff da opção de compra
    return Sn, k, payoff

Sn, k, payoff = simulacao(n_steps, p, u, d, s0, strike)
print(f"Sn = {Sn:.4f} | k = {k} | payoff = {payoff:.4f}")

### Preço teórico (fórmula fechada binomial)Somando sobre **todos** os cenários possíveis de $k$ (de 0 a $n_{steps}$), ponderados pela sua probabilidade exata, obtemos o valor esperado do payoff — que, descontado, é o preço justo $C$:$$C = e^{-rT} \sum_{k=0}^{n} \binom{n}{k} p^k (1-p)^{n-k} \max(S_0 u^k d^{n-k} - K,\ 0)$$Este $C$ é o "gabarito" contra o qual comparamos todas as estimativas via simulação nos módulos seguintes.

In [ ]:
esperanca = 0
for l in range(n_steps + 1):                        # percorre TODOS os cenários k = 0..n_steps
    probk = binom.pmf(l, n_steps, p)                # probabilidade de exatamente l subidas
    Snk = s0 * (u ** l) * (d ** (n_steps - l))      # preço da ação nesse cenário
    payoffk = max(Snk - strike, 0)                  # payoff nesse cenário
    esperanca += probk * payoffk                    # contribuição ponderada

def teorico(j, t, e):
    return M.exp(-j * t) * e                         # aplica o desconto

C = teorico(r, T, esperanca)
print(f"Preço teórico (C): {C:.4f}")

## Módulo 1 — Lei Fraca dos Grandes Números (LFGN)A LFGN afirma que a média amostral converge **em probabilidade** para o valor esperado: para qualquer margem $\varepsilon$, a probabilidade de a estimativa $\hat{C}_N$ estar longe do valor real $C$ tende a zero conforme $N$ cresce.$$P\left(|\hat{C}_N - C| > \varepsilon\right) \xrightarrow{N\to\infty} 0$$**Estratégia:** para cada tamanho de amostra $N$, geramos $M$ estimativas independentes $\hat{C}_N$ e medimos a fração delas que erra por mais de $\varepsilon$.

In [ ]:
simulacao_N = [10, 100, 1000, 10000]   # tamanhos de amostra a testar
m_repeticao = 1000                      # nº de estimativas independentes por N
epsilon = 0.20                          # margem de erro considerada "grande"

def estimador_preco(N):
    payoff_N = []
    for _ in range(N):
        _, _, pyf = simulacao(n_steps, p, u, d, s0, strike)
        payoff_N.append(pyf)
    return M.exp(-r * T) * np.mean(payoff_N)   # estimativa Ĉ_N (média descontada)

# gera M estimativas para cada N
resultados = {}
for N in simulacao_N:
    resultados[N] = [estimador_preco(N) for _ in range(m_repeticao)]

# fração de estimativas que erram por mais de epsilon
margem_erro = {}
for N in simulacao_N:
    erros = np.sum(np.abs(np.array(resultados[N]) - C) > epsilon)
    margem_erro[N] = erros / m_repeticao

print("Fração de erro (> epsilon) por N:")
for N, frac in margem_erro.items():
    print(f"  N = {N:>6} -> {frac:.3f}  ({frac*100:.1f}%)")

### Gráfico — a probabilidade de erro cai para zero

In [ ]:
x = list(margem_erro.keys())
y = list(margem_erro.values())

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=120)
ax.plot(x, y, marker='o', markersize=9, linewidth=2.5, color='#2E5EAA',
        markerfacecolor='#2E5EAA', markeredgecolor='white', markeredgewidth=1.5, zorder=3)

ax.axhline(y=0.05, color='#D64550', linestyle='--', linewidth=1.3, alpha=0.7, zorder=1)
ax.text(x[0], 0.07, 'limiar de 5%', color='#D64550', fontsize=9, va='bottom')

for xi, yi in zip(x, y):
    ax.annotate(f'{yi*100:.1f}%', xy=(xi, yi), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=9,
                fontweight='bold', color='#2E5EAA')

ax.set_xscale('log')
ax.set_xlabel('Número de simulações (N)', fontsize=11)
ax.set_ylabel(r'$P(|\hat{C}_N - C| > \varepsilon)$', fontsize=12)
ax.set_title('Convergência em probabilidade (LFGN)\nEstimador de Monte Carlo do preço da opção',
             fontsize=13, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(-0.05, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Módulo 2 — Lei Forte dos Grandes Números (convergência quase certa)A Lei Forte é uma afirmação mais forte: com probabilidade 1, a **trajetória** da média amostral converge para o valor real.$$P\left(\lim_{n\to\infty} \hat{C}_n = C\right) = 1$$**Estratégia:** em vez de repetir o experimento, acompanhamos a **média acumulada** dentro de uma única sequência longa de simulações, e mostramos que várias trajetórias independentes convergem suavemente para $C$.

In [ ]:
c = 100000   # comprimento de cada trajetória (nº de simulações acumuladas)

# coleta os payoffs de uma trajetória
payoff_soma = [simulacao(n_steps, p, u, d, s0, strike)[2] for _ in range(c)]

soma = np.cumsum(payoff_soma)        # soma acumulada
deno = np.arange(1, c + 1)           # divisores: 1, 2, 3, ..., c
Xn = (soma / deno) * M.exp(-r * T)   # média acumulada, descontada

print(f"Estimativa após 1 simulação:      {Xn[0]:.4f}")
print(f"Estimativa após {c} simulações:  {Xn[-1]:.4f}")
print(f"Preço teórico (C):                {C:.4f}")

### Gráfico — múltiplas trajetórias convergindo para C

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=120)

n_trajetorias = 5
cores = plt.cm.viridis(np.linspace(0, 0.85, n_trajetorias))

for idx in range(n_trajetorias):
    payoff_traj = [simulacao(n_steps, p, u, d, s0, strike)[2] for _ in range(c)]
    soma = np.cumsum(payoff_traj)
    Xn_traj = (soma / deno) * M.exp(-r * T)
    ax.plot(deno, Xn_traj, linewidth=1.3, alpha=0.85, color=cores[idx], label=f'Trajetória {idx+1}')

ax.axhline(y=C, color='#D64550', linestyle='--', linewidth=2,
           label=f'Preço teórico (C ≈ {C:.2f})', zorder=5)

ax.set_xscale('log')
ax.set_xlabel('Número de simulações acumuladas (n)', fontsize=11)
ax.set_ylabel(r'$\hat{C}_n$ (preço estimado)', fontsize=12)
ax.set_title('Convergência quase certa (Lei Forte)\nMédia acumulada de múltiplas trajetórias independentes',
             fontsize=13, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Módulo 3 — Teorema Central do Limite (TLC)O TLC aparece em dois níveis neste projeto.### Nível 1 — Teorema de De Moivre–LaplaceA distribuição Binomial (o número de subidas $k$), quando padronizada, converge para a Normal padrão conforme $n_{steps}$ cresce:$$Z = \frac{k - np}{\sqrt{np(1-p)}} \xrightarrow{d} N(0,1)$$

In [ ]:
steps = [10, 50, 100, 500, 1000, 5000]   # nº de passos a testar
n_amostras = 100000                        # sorteios por valor de n_steps

k_variados = {}
for g in steps:
    sort = np.random.binomial(g, p, size=n_amostras)          # sorteia muitos k
    z = (sort - g * p) / M.sqrt(g * p * (1 - p))              # padroniza (De Moivre-Laplace)
    k_variados[g] = z

print("Arrays padronizados gerados para n_steps =", steps)

#### Gráfico — a Binomial padronizada vira Normal

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), dpi=110)
axes = axes.flatten()

x = np.linspace(-4, 4, 300)
normal_pdf = norm.pdf(x)

for idx, g in enumerate(steps):
    ax = axes[idx]
    ax.hist(k_variados[g], bins=50, density=True, color='#2E5EAA', alpha=0.65,
            edgecolor='white', linewidth=0.3, label='Binomial padronizada')
    ax.plot(x, normal_pdf, color='#D64550', linewidth=2.2, label='Normal N(0,1)')
    ax.set_title(f'n_steps = {g}', fontsize=12, fontweight='bold')
    ax.set_xlim(-4, 4)
    ax.set_xlabel('Z (valor padronizado)', fontsize=9)
    ax.set_ylabel('Densidade', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Teorema de De Moivre–Laplace\nA distribuição Binomial padronizada converge para a Normal conforme n_steps cresce',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Nível 2 — TLC aplicado ao estimador de Monte CarloMesmo que os payoffs individuais tenham distribuição "torta" (muitos zeros, alguns valores altos), a **média** deles ($\hat{C}_N$) segue aproximadamente uma Normal — é o que permite construir intervalos de confiança.Primeiro, verificamos que a distribuição das estimativas $\hat{C}_N$ (geradas no Módulo 1) tem formato de sino.

In [ ]:
# padroniza a distribuição das estimativas de cada N (dados do Módulo 1)
Cn_padronizado = {}
for N in simulacao_N:
    arr = np.array(resultados[N])
    Cn_padronizado[N] = (arr - arr.mean()) / arr.std()

fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=110)
axes = axes.flatten()
x = np.linspace(-4, 4, 300)
normal_pdf = norm.pdf(x)

for idx, N in enumerate(simulacao_N):
    ax = axes[idx]
    ax.hist(Cn_padronizado[N], bins=40, density=True, color='#2E5EAA', alpha=0.65,
            edgecolor='white', linewidth=0.3, label=r'$\hat{C}_N$ padronizado')
    ax.plot(x, normal_pdf, color='#D64550', linewidth=2.2, label='Normal N(0,1)')
    ax.set_title(f'N = {N} simulações', fontsize=12, fontweight='bold')
    ax.set_xlim(-4, 4)
    ax.set_xlabel('Z (estimativa padronizada)', fontsize=9)
    ax.set_ylabel('Densidade', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('TLC aplicado ao estimador de Monte Carlo\n'
             r'A distribuição de $\hat{C}_N$ se aproxima da Normal mesmo com payoffs não-normais',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Intervalos de confiança e validação da coberturaA partir de **uma única** estimativa, o TLC permite construir um intervalo de confiança de 95%:$$\hat{C}_N \pm 1{,}96 \cdot \frac{\hat{\sigma}}{\sqrt{N}}$$onde $\hat{\sigma}$ é o desvio padrão dos payoffs. Para validar, repetimos a construção muitas vezes e verificamos se ~95% dos intervalos realmente contêm o valor teórico $C$.

In [ ]:
def estimador_com_IC(N):
    payoffs = []
    for _ in range(N):
        _, _, pyf = simulacao(n_steps, p, u, d, s0, strike)
        payoffs.append(pyf)

    payoffs_desc = np.array(payoffs) * M.exp(-r * T)   # payoffs descontados
    Cn = payoffs_desc.mean()                            # estimativa pontual
    desvio = payoffs_desc.std()                         # desvio dos payoffs
    margem = 1.96 * desvio / np.sqrt(N)                 # metade do intervalo (95%)
    return Cn, Cn - margem, Cn + margem

# validação empírica da cobertura
N_fixo = 1000
M_cobertura = 1000
acertos = 0
intervalos = []

for _ in range(M_cobertura):
    estimativa, inf, sup = estimador_com_IC(N_fixo)
    intervalos.append((estimativa, inf, sup))
    if inf <= C <= sup:
        acertos += 1

cobertura_empirica = acertos / M_cobertura
print(f"Cobertura empírica:  {cobertura_empirica:.3f}  ({cobertura_empirica*100:.1f}%)")
print(f"Cobertura esperada:  0.950  (95.0%)")
print(f"Intervalos que capturaram C: {acertos} de {M_cobertura}")

#### Gráfico — os intervalos que acertam (verde) e os que erram (vermelho)

In [ ]:
from matplotlib.lines import Line2D

n_mostrar = 50
subset = intervalos[:n_mostrar]

fig, ax = plt.subplots(figsize=(12, 7), dpi=120)
for idx, (estimativa, inf, sup) in enumerate(subset):
    capturou = inf <= C <= sup
    cor = '#2E9E5B' if capturou else '#D64550'
    ax.plot([idx, idx], [inf, sup], color=cor, linewidth=1.8, alpha=0.8, zorder=2)
    ax.plot(idx, estimativa, 'o', color=cor, markersize=4, zorder=3)

ax.axhline(y=C, color='#1A1A2E', linestyle='--', linewidth=2,
           label=f'Preço teórico (C ≈ {C:.3f})', zorder=1)

erros = sum(1 for (est, inf, sup) in subset if not (inf <= C <= sup))
ax.set_xlabel('Repetição do experimento (cada linha = 1 intervalo de confiança)', fontsize=11)
ax.set_ylabel(r'$\hat{C}_N$ e intervalo de 95%', fontsize=12)
ax.set_title(f'Validação da cobertura do intervalo de confiança (TLC)\n'
             f'Cobertura empírica geral: {cobertura_empirica*100:.1f}%  |  '
             f'Neste recorte de {n_mostrar}: {n_mostrar - erros} acertos, {erros} erros',
             fontsize=13, fontweight='bold', pad=15)

legend_elements = [
    Line2D([0], [0], color='#2E9E5B', lw=2, label='Capturou C (acerto)'),
    Line2D([0], [0], color='#D64550', lw=2, label='Não capturou C (erro)'),
    Line2D([0], [0], color='#1A1A2E', lw=2, linestyle='--', label=f'C ≈ {C:.3f}')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Módulo 4 — Convergência do Binomial para Black–ScholesFechamento conceitual: o modelo binomial (discreto) converge para o modelo de **Black–Scholes** (contínuo) conforme $n_{steps} \to \infty$. Ou seja, chegamos ao mesmo resultado do modelo contínuo — que exigiria cálculo estocástico — usando apenas probabilidade e combinatória.A fórmula fechada de Black–Scholes para uma call europeia:$$C_{BS} = S_0\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2), \qquadd_1 = \frac{\ln(S_0/K) + (r + \tfrac{\sigma^2}{2})T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
def preco_binomial(N):
    delta = T / N
    us = M.exp(dp_anual * M.sqrt(delta))
    md_ = 1 / us
    mp = (M.exp(r * delta) - md_) / (us - md_)

    hope = 0
    for t in range(N + 1):
        probt = binom.pmf(t, N, mp)
        Snt = s0 * (us ** t) * (md_ ** (N - t))
        hope += probt * max(Snt - strike, 0)
    return M.exp(-r * T) * hope

def black_scholes():
    d1 = (M.log(s0 / strike) + (r + (dp_anual ** 2) / 2) * T) / (dp_anual * M.sqrt(T))
    d2 = d1 - dp_anual * M.sqrt(T)
    return s0 * norm.cdf(d1) - strike * M.exp(-r * T) * norm.cdf(d2)

print(f"Binomial (N=10):   {preco_binomial(10):.4f}")
print(f"Binomial (N=500):  {preco_binomial(500):.4f}")
print(f"Black-Scholes:     {black_scholes():.4f}")

### Gráfico — o Binomial converge para Black–Scholes

In [ ]:
valores_N = [2, 3, 5, 8, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300, 500]
precos_bin = [preco_binomial(N) for N in valores_N]
preco_bs = black_scholes()

fig, ax = plt.subplots(figsize=(12, 7), dpi=130)

tolerancia = preco_bs * 0.005
ax.axhspan(preco_bs - tolerancia, preco_bs + tolerancia, color='#D64550', alpha=0.10, zorder=0)
ax.fill_between(valores_N, precos_bin, preco_bs, color='#2E5EAA', alpha=0.12, zorder=1)
ax.axhline(y=preco_bs, color='#D64550', linestyle='--', linewidth=2.2,
           label=f'Black–Scholes (limite contínuo) = {preco_bs:.4f}', zorder=3)

cores_pontos = plt.cm.viridis(np.linspace(0.15, 0.85, len(valores_N)))
ax.plot(valores_N, precos_bin, linewidth=2.3, color='#2E5EAA', alpha=0.85, zorder=4,
        label='Preço Binomial (discreto)')
ax.scatter(valores_N, precos_bin, s=70, c=cores_pontos, edgecolors='white',
           linewidths=1.4, zorder=5)

ax.annotate(f'N={valores_N[0]}\n{precos_bin[0]:.2f}', xy=(valores_N[0], precos_bin[0]),
            xytext=(15, 25), textcoords='offset points', fontsize=9, fontweight='bold',
            color='#2E5EAA', arrowprops=dict(arrowstyle='->', color='#2E5EAA', alpha=0.6))
ax.annotate(f'N={valores_N[-1]}\n{precos_bin[-1]:.4f}', xy=(valores_N[-1], precos_bin[-1]),
            xytext=(-55, -38), textcoords='offset points', fontsize=9, fontweight='bold',
            color='#2E7D5B', arrowprops=dict(arrowstyle='->', color='#2E7D5B', alpha=0.6))

ax.set_xscale('log')
ax.set_xlabel('Número de passos da árvore binomial  (N)  —  escala log', fontsize=11.5)
ax.set_ylabel('Preço da opção de compra (R$)', fontsize=11.5)
ax.set_title('Convergência do Modelo Binomial ao Black–Scholes', fontsize=15, fontweight='bold', pad=22)
ax.text(0.5, 1.015, 'O modelo discreto reproduz o contínuo no limite — sem cálculo estocástico',
        transform=ax.transAxes, ha='center', fontsize=10.5, color='#555555', style='italic')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')
ax.legend(loc='lower right', fontsize=10.5, framealpha=0.95, edgecolor='#DDDDDD', fancybox=True)
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.6)
ax.set_xticks([2, 5, 10, 20, 50, 100, 200, 500])
ax.set_xticklabels(['2', '5', '10', '20', '50', '100', '200', '500'])
plt.tight_layout()
plt.show()

## ConclusãoUsando a precificação de uma opção de compra da PETR4 como laboratório, demonstramos empiricamente os três pilares da teoria dos grandes números:- **Módulo 1 (LFGN):** a probabilidade de a estimativa $\hat{C}_N$ errar por mais de $\varepsilon$ cai de forma acentuada conforme $N$ cresce, tendendo a zero — convergência em probabilidade.- **Módulo 2 (Lei Forte):** cada trajetória individual da média acumulada converge suavemente para o preço teórico $C$, sem reverter — convergência quase certa.- **Módulo 3 (TLC):** a distribuição das estimativas se aproxima da Normal (mesmo com payoffs não-normais), e os intervalos de confiança de 95% construídos a partir disso capturam o valor real na proporção esperada — cobertura empírica próxima de 95%.- **Módulo 4:** o modelo binomial discreto converge para o modelo contínuo de Black–Scholes conforme o número de passos cresce, unindo o caminho combinatório ao resultado clássico das finanças quantitativas.O ponto central é que esses teoremas — normalmente abstratos — governam diretamente a precisão e a confiabilidade de qualquer estimativa obtida por simulação de Monte Carlo.